# KNN Classifier — Hotel Booking Cancellation Prediction

Author: Siriwardana N.D.V.S

KNN (K-Nearest Neighbors) is a supervised, non-parametric, instance-based lazy learner. Unlike other models, it stores the entire training set and classifies new bookings by majority vote among the K most similar training examples using Euclidean or Manhattan distance.

The objective of this notebook is to predict whether a hotel booking will be cancelled (`is_canceled = 1`) or not (`is_canceled = 0`) using the Hotel Booking Demand dataset, and compare KNN performance against the group's other models (Logistic Regression, Decision Tree, Random Forest).

In [3]:
import os
import sys

current_dir = os.path.abspath(os.getcwd())
project_root = None

for _ in range(6):
    config_path = os.path.join(current_dir, "src", "config.py")
    if os.path.exists(config_path):
        project_root = current_dir
        break
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

if project_root is None:
    raise RuntimeError(
        "Could not find src/config.py within 5 parent levels. "
        "Open VS Code from the project root and rerun this cell."
    )

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: d:\SLIIT\Year_4_Sem_2\Z_Projects\ml-assignment


In [4]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance
import sklearn

from src import config
from src.data_loader import load_hotel_bookings, basic_train_ready_checks
from src.preprocessing import build_preprocessor, PreprocessOptions, get_feature_names
from src.train_eval import (
    TrainOptions, split_xy, make_train_test_split,
    get_estimator, build_model_pipeline, tune_with_gridsearch,
    predict_with_optional_proba, evaluate_on_test
    )
from src.metrics import compute_classification_metrics, format_metrics_for_print
from src.plots import plot_confusion_matrix, plot_roc_curve, plot_pr_curve
from src.io_utils import (ensure_artifact_dirs, save_json, save_text,
                          save_dataframe, save_model, save_run_metadata)

DIRS = ensure_artifact_dirs()
print("Artifact directories:")
for name, path in DIRS.items():
    print(f"- {name}: {path}")

print(f"Python version: {sys.version}")
print(f"scikit-learn version: {sklearn.__version__}")

Artifact directories:
- base: artifacts
- data: artifacts\data
- preprocessing: artifacts\preprocessing
- models: artifacts\models
- metrics: artifacts\metrics
- plots: artifacts\plots
- reports: artifacts\reports
Python version: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
scikit-learn version: 1.7.2


## 1. Data Loading

The dataset is loaded using the project's `data_loader` utility, preferring the deduplicated processed version if available, with a fallback to the raw dataset path from config.

In [5]:
processed_path = os.path.join("data", "processed", "hotel_bookings_dedup.csv")

if os.path.exists(processed_path):
    df = load_hotel_bookings(processed_path, drop_duplicates=False, verbose=True)
    print("Loaded deduplicated dataset from processed folder")
else:
    df = load_hotel_bookings(config.DEFAULT_DATA_PATH, drop_duplicates=True, verbose=True)
    print("Processed file not found, loaded from raw path with deduplication")

basic_train_ready_checks(df, target_col="is_canceled")

for col in ["agent", "company"]:
    if col in df.columns:
        df[col] = df[col].astype(str)

print(f"Dataset shape: {df.shape}")

class_counts = df["is_canceled"].value_counts(dropna=False).sort_index()
class_perc = (df["is_canceled"].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)

print("Class distribution (is_canceled):")
for label in class_counts.index:
    print(f"  {label}: {int(class_counts[label])} ({class_perc[label]:.2f}%)")

print("Confirmed: 'agent' and 'company' cast to str")

[data_loader] Loaded shape: (87396, 32)
[data_loader] Columns: 32
Loaded deduplicated dataset from processed folder
Dataset shape: (87396, 32)
Class distribution (is_canceled):
  0: 63371 (72.51%)
  1: 24025 (27.49%)
Confirmed: 'agent' and 'company' cast to str


In [6]:
X, y = split_xy(df, target_col="is_canceled")

opts = TrainOptions(test_size=0.20, random_state=42)
X_train, X_test, y_train, y_test = make_train_test_split(X, y, options=opts)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape} | y_test shape: {y_test.shape}")

train_counts = y_train.value_counts(dropna=False).sort_index()
train_perc = (y_train.value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)
test_counts = y_test.value_counts(dropna=False).sort_index()
test_perc = (y_test.value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)

print("Class distribution in y_train:")
for label in train_counts.index:
    print(f"  {label}: {int(train_counts[label])} ({train_perc[label]:.2f}%)")

print("Class distribution in y_test:")
for label in test_counts.index:
    print(f"  {label}: {int(test_counts[label])} ({test_perc[label]:.2f}%)")

train_dist = {str(k): int(v) for k, v in y_train.value_counts().to_dict().items()}
test_dist = {str(k): int(v) for k, v in y_test.value_counts().to_dict().items()}

split_metadata = {
    "model": "knn",
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "test_ratio": 0.20,
    "random_state": 42,
    "stratified": True,
    "train_class_distribution": train_dist,
    "test_class_distribution": test_dist,
}

split_meta_path = os.path.join("artifacts", "data", "train_test_split_knn.json")
save_json(split_metadata, split_meta_path)
print("Split metadata saved to artifacts/data/train_test_split_knn.json")

X_train shape: (69916, 31) | y_train shape: (69916,)
X_test shape: (17480, 31) | y_test shape: (17480,)
Class distribution in y_train:
  0: 50696 (72.51%)
  1: 19220 (27.49%)
Class distribution in y_test:
  0: 12675 (72.51%)
  1: 4805 (27.49%)
Split metadata saved to artifacts/data/train_test_split_knn.json
